# Testes do pipeline de Agenda

Notebook didático e objetivo para testar as funcionalidades da pasta `src/agenda`.

Fluxo coberto:
- Pré-processamento de arquivos `.txt` da agenda política;
- Geração de embeddings semânticos por chunks;
- Inspeção rápida de resultados.

In [9]:
import sys
from pathlib import Path
import pandas as pd

# ------------------------------------------------------------------
# Descobre a raiz do projeto (pasta que contém ./src)
# Isso evita problemas de import dependendo de onde o notebook foi aberto.
# ------------------------------------------------------------------
def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists():
            return candidate
    raise RuntimeError("Não foi possível localizar a raiz do projeto (pasta com ./src).")

PROJECT_ROOT = find_project_root(Path.cwd())
AGENDA_SRC = PROJECT_ROOT / "src" / "agenda"

# Adiciona src/agenda no path para importar os módulos locais.
if str(AGENDA_SRC) not in sys.path:
    sys.path.append(str(AGENDA_SRC))

from pre_processing import processar_todos_elementos
from embeddings import generate_agenda_embeddings_from_txt

print("PROJECT_ROOT:", PROJECT_ROOT)
print("AGENDA_SRC:", AGENDA_SRC)

PROJECT_ROOT: /home/gpcmoura/Documents/Code/Msc/Repositories/political-discourse-analysis
AGENDA_SRC: /home/gpcmoura/Documents/Code/Msc/Repositories/political-discourse-analysis/src/agenda


## 1) Pré-processamento da agenda

A célula abaixo processa os `.txt` de um elemento específico (modo teste) ou de todos os elementos.

- Para teste rápido, use `elemento_teste = "UNIAO"` (ou outro elemento existente).
- Para processar tudo, use `elemento_teste = None`.

In [11]:
# Caminhos de entrada e saída do pré-processamento
pasta_agenda_politica = PROJECT_ROOT / "data" / "party_agenda" / "party"
pasta_saida_tokens = PROJECT_ROOT / "data" / "party_agenda" / "preprocessing" / "tokenization" / "tokens"
pasta_saida_tokens.mkdir(parents=True, exist_ok=True)

# Ajuste para teste rápido
elemento_teste = "UNIAO"  # ex.: "UNIAO" | use None para processar todos

# Executa o pipeline de pré-processamento
processar_todos_elementos(
    pasta_agenda_politica=pasta_agenda_politica,
    pasta_saida_tokens=pasta_saida_tokens,
    elemento_teste=elemento_teste,
)


[Elemento] UNIAO
  - Manifesto_Uniao_BRASIL.layout_aware.txt -> Manifesto_Uniao_BRASIL.layout_aware_tokens.txt

Concluído: 1 arquivo(s) processado(s) para o elemento UNIAO.


In [6]:
# Inspeção simples dos arquivos gerados
elemento_para_inspecao = elemento_teste if elemento_teste else "UNIAO"
base_saida = pasta_saida_tokens / elemento_para_inspecao

txt_files = sorted((base_saida / "TXT").glob("*.txt")) if (base_saida / "TXT").exists() else []
csv_files = sorted((base_saida / "CSV").glob("*.csv")) if (base_saida / "CSV").exists() else []

print("Pasta de inspeção:", base_saida)
print("Qtd TXT de tokens:", len(txt_files))
print("Qtd CSV preprocessados:", len(csv_files))

if csv_files:
    display(pd.read_csv(csv_files[0]).head(1))
else:
    print("Nenhum CSV encontrado para inspeção.")

Pasta de inspeção: /home/gpcmoura/Documents/Code/Msc/Repositories/political-discourse-analysis/data/preprocessing/tokenization/tokens/UNIAO
Qtd TXT de tokens: 1
Qtd CSV preprocessados: 1


,preprocess_agenda,tokens
0,valor princípio brasil reconhecer celebrar plu...,"[""valor"", ""princípio"", ""brasil"", ""reconhecer"",..."


## 2) Embeddings da agenda por chunks semânticos

A célula abaixo testa a função `generate_agenda_embeddings_from_txt` com um `.txt` da agenda.

Arquivo sugerido para teste:
`data/party_agenda/party/UNIAO/txt/Manifesto_Uniao_BRASIL.layout_aware.txt`

Saída esperada dos embeddings:
`data/party_agenda/embeddings/<PARTIDO>/`

In [ ]:
# Ajuste aqui o arquivo de entrada
txt_path = PROJECT_ROOT / "data" / "party_agenda" / "party" / "UNIAO" / "txt" / "Manifesto_Uniao_BRASIL.layout_aware.txt"

# Define o partido com base no caminho do arquivo (..../party/<PARTIDO>/txt/arquivo.txt)
partido_atual = txt_path.parent.parent.name

# Pasta de saída dos embeddings por partido
output_dir = PROJECT_ROOT / "data" / "party_agenda" / "embeddings" / partido_atual
output_dir.mkdir(parents=True, exist_ok=True)

emb_df, emb_matrix, base_name = generate_agenda_embeddings_from_txt(
    txt_path=str(txt_path),
    similarity_threshold=0.45,
    min_sentences_per_chunk=1,
    max_sentences_per_chunk=None,
    batch_size=32,
    output_dir=str(output_dir),
    save_files=True,
)

print("Partido:", partido_atual)
print("Output dir:", output_dir)
print("Base name:", base_name)
print("Chunks gerados:", len(emb_df))
print("Shape embeddings:", emb_matrix.shape)
display(emb_df[["source_id", "chunk_id", "chunk_text"]].head(5))

............................................
... Função generate_text_embeddings iniciada ...


/home/gpcmoura/Documents/Code/Msc/Repositories/political-discourse-analysis/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8545.78it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


... Total de chunks gerados: 105 ...


Batches: 100%|██████████| 4/4 [00:01<00:00,  2.11it/s]

... Arquivo CSV salvo em: /home/gpcmoura/Documents/Code/Msc/Repositories/political-discourse-analysis/data/running_files/agenda_embeddings_Manifesto_Uniao_BRASIL.layout_aware.txt_20260411_1339.csv ...
... Arquivo NPY salvo em: /home/gpcmoura/Documents/Code/Msc/Repositories/political-discourse-analysis/data/running_files/agenda_embeddings_Manifesto_Uniao_BRASIL.layout_aware.txt_20260411_1339.npy ...
... Função generate_text_embeddings encerrada ...
..............................................
Base name: agenda_embeddings_Manifesto_Uniao_BRASIL.layout_aware.txt_20260411_1339
Chunks gerados: 105
Shape embeddings: (105, 384)


,source_id,chunk_id,chunk_text
0,Manifesto_Uniao_BRASIL.layout_aware.txt,0,Nossos Valores e Princípios O Brasil é reconhe...
1,Manifesto_Uniao_BRASIL.layout_aware.txt,1,"São muitas e diferentes as nossas origens, as ..."
2,Manifesto_Uniao_BRASIL.layout_aware.txt,2,É ele que nunca deixa de pulsar em nossos cora...
3,Manifesto_Uniao_BRASIL.layout_aware.txt,3,Esse Amor pelo Brasil está assentado na nossa ...
4,Manifesto_Uniao_BRASIL.layout_aware.txt,4,Mas não é uma utopia.


In [ ]:
# Inspeção manual de alguns chunks
n_amostras = 3
for i, chunk in enumerate(emb_df["chunk_text"].head(n_amostras), start=1):
    print(f"\n--- Chunk {i} ---")
    print(chunk[:1200])  # limita visualização para não poluir a saída

## 3) Tópicos (placeholder)

No momento, o arquivo `src/agenda/topics.py` está sem implementação.

Quando esse módulo for implementado, inclua aqui células análogas para:
- carregar os dados preprocessados;
- executar modelagem de tópicos;
- inspecionar termos e distribuição por tópico.

In [ ]:
# Verificação rápida do estado do módulo de tópicos
topics_file = PROJECT_ROOT / "src" / "agenda" / "topics.py"
content = topics_file.read_text(encoding="utf-8", errors="ignore").strip()
print("Arquivo:", topics_file)
print("Caracteres no arquivo topics.py:", len(content))
if len(content) <= 10:
    print("topics.py aparentemente ainda não implementado.")
else:
    print("topics.py possui conteúdo. Você pode adicionar testes nesta seção.")